## Inverse Velocity Kinematics - Lab 1 R-R-P robot
Given a wanted tip velocity $v = \dot p_e$, find the joint velocities $\dot q$.

The robot has 3 joints, so only the tip **linear** velocity can be chosen (3 equations, 3 unknowns).
Use the $3\times3$ linear part $J_v$ from `fvk.ipynb`:

$v = J_v(q)\,\dot q \quad\Rightarrow\quad \dot q = J_v^{-1}(q)\, v \qquad (\text{only if } \det J_v \ne 0)$

In [ ]:
import numpy as np
import sympy as sp
from IPython.display import display, Math
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

def Rx(t): return sp.Matrix([[1, 0, 0, 0], [0, sp.cos(t), -sp.sin(t), 0], [0, sp.sin(t), sp.cos(t), 0], [0, 0, 0, 1]])
def Rz(t): return sp.Matrix([[sp.cos(t), -sp.sin(t), 0, 0], [sp.sin(t), sp.cos(t), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tx(d): return sp.Matrix([[1, 0, 0, d], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
def Tz(d): return sp.Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, d], [0, 0, 0, 1]])

# Modified DH (Craig): T^{i-1}_i = Rx(alpha_{i-1}) Tx(a_{i-1}) Rz(theta_i) Tz(d_i)
def mdh(a, alpha, d, theta):
    return Rx(alpha) * Tx(a) * Rz(theta) * Tz(d)

# display only: sin(q1) -> s_1, cos(q1 + q2) -> c_12
def short(e):
    def sub(f):
        terms = sp.Add.make_args(f.args[0])
        if not all(t.is_Symbol and t.name[0] == 'q' for t in terms):
            return f
        idx = ''.join(sorted((t.name[1:] for t in terms), key=int))
        return sp.Symbol(('s' if f.func == sp.sin else 'c') + '_{' + idx + '}')
    return e.replace(lambda t: t.func in (sp.sin, sp.cos), sub)

def show(lhs, rhs):
    tex = sp.latex(short(rhs)).replace(r'\operatorname{atan}_{2}', r'\operatorname{atan2}')
    display(Math(lhs + ' = ' + tex))

### 0 - Linear Jacobian $J_v$
Same as `fvk.ipynb` step 1.

In [ ]:
q1, q2, q3 = sp.symbols('q1:4', real=True)
l1, l2, l3, l4 = sp.symbols('l1:5', positive=True)

# MDH table -> (a_{i-1}, alpha_{i-1}, d_i, theta_i)
mdh_table = [
    (0, 0,        l1,      q1),
    (0, sp.pi/2, -l2,      q2 + sp.pi/2),
    (0, sp.pi/2,  q3 + l3, 0),
]
T_tool = Tz(l4)
q = sp.Matrix([q1, q2, q3])

T0e = sp.eye(4)
for row in mdh_table:
    T0e = T0e * mdh(*row)
T0e = sp.simplify(T0e * T_tool)
p = T0e[0:3, 3]

Jv = sp.simplify(p.jacobian(q))
show(r'J_v', Jv)

### 1 - Singularities: $\det J_v = 0$
At a singularity $J_v$ has no inverse: some tip directions cannot be reached at any joint speed.

In [ ]:
det_Jv = sp.factor(sp.simplify(Jv.det()))
show(r'\det J_v', det_Jv)

$\det J_v = 0$ when
- $c_2 = 0$ ($q_2 = \pm 90°$): the arm points straight up/down. Joints 1 and 2 then both move the tip in the same horizontal direction, so the other horizontal direction is lost.
- $L = l_3 + q_3 + l_4 = 0$: the tip is on joint 2 (not physically possible here).

### 2 - Inverse Jacobian and the IVK equation

In [ ]:
v = sp.Matrix(sp.symbols('v_x v_y v_z', real=True))
Jv_inv = sp.simplify(Jv.inv())

show(r'J_v^{-1}', Jv_inv)
show(r'\dot q = J_v^{-1} v', sp.simplify(Jv_inv * v))

### 3 - Check with numbers
Pick a pose and a wanted tip velocity, solve $\dot q$, then put $\dot q$ back into FVK: $J_v \dot q$ should equal $v$.
The last test pose is singular ($q_2 = 90°$) on purpose.

In [ ]:
# edit here: link lengths (m)
vals = {l1: 0.64, l2: 0.2, l3: 0.22, l4: 0.095}

robot_mdh = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=vals[l1]),
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-vals[l2], offset=np.pi/2),
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=vals[l3]),
], name='my_mdh')
robot_mdh.tool = SE3.Tz(vals[l4])

Jv_inv_f = sp.lambdify((q1, q2, q3), Jv_inv.subs(vals), 'numpy')

# edit here: wanted tip velocity (m/s) and poses to test
v_want = np.array([0.05, -0.02, 0.03])
for q_test in ([np.pi/6, np.pi/6, 0.18], [np.deg2rad(-25), np.deg2rad(-25), 0.135], [0.3, np.pi/2, 0.1]):
    q_test = np.array(q_test)
    Jv_rtb = robot_mdh.jacob0(q_test)[:3]          # linear rows of the RTB Jacobian
    det = np.linalg.det(Jv_rtb)
    print('q =', np.round(q_test, 4), f'| det(Jv) = {det:.2e}')
    if abs(det) < 1e-9:
        print('  singular -> no unique q_dot')
        continue
    qd = np.array(Jv_inv_f(*q_test), dtype=float) @ v_want
    err = np.linalg.norm(Jv_rtb @ qd - v_want)
    print('  q_dot =', np.round(qd, 4), f'| |Jv q_dot - v| = {err:.1e}')

### Numeric example
Plug a pose and a wanted tip velocity into $J_v$, $\det J_v$, $J_v^{-1}$ and $\dot q = J_v^{-1} v$.

In [ ]:
# helpers: plug numbers into a symbolic result and print it rounded
def val(e, subs):
    return float(sp.sympify(e).subs(subs).evalf())
def mat(e, subs, d=4):
    return sp.Matrix(e).subs(subs).evalf().applyfunc(lambda v: sp.Float(round(float(v), d)) if abs(float(v)) > 1e-12 else 0)
def showv(lhs, v, unit=''):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{{unit}}}'))
def showa(lhs, v):
    display(Math(rf'{lhs} = {v:.4f}\ \text{{rad}} = {np.rad2deg(v):.2f}^\circ'))
def showm(lhs, M):
    display(Math(lhs + ' = ' + sp.latex(M)))

# edit here: joint values (deg, deg, m) and wanted tip velocity (m/s)
q_in = [30, 30, 0.18]
v_in = [0.05, -0.02, 0.03]

q_val = [np.deg2rad(q_in[0]), np.deg2rad(q_in[1]), q_in[2]]
nums = {**vals, q1: q_val[0], q2: q_val[1], q3: q_val[2]}

Jv_num = mat(Jv, nums)
showm(r'J_v', Jv_num)
detv = val(det_Jv, nums)
showv(r'\det J_v', detv)
if abs(detv) < 1e-9:
    print('singular -> no unique q_dot')
else:
    Jinv_num = mat(Jv_inv, nums)
    showm(r'J_v^{-1}', Jinv_num)
    qd_num = mat(Jinv_num * sp.Matrix(v_in), {})
    showm(r'\dot q = J_v^{-1} v', qd_num)
    print('units: rad/s, rad/s, m/s')
    err = np.linalg.norm(robot_mdh.jacob0(q_val)[:3] @ np.array((Jv_inv * sp.Matrix(v_in)).subs(nums).evalf(), dtype=float).flatten() - v_in)
    print(f'check: |Jv q_dot - v| = {err:.1e}')

### Same with Robotics Toolbox
Uses the same `q_in`, `v_in` as the numeric example. `jacob0(q)[:3]` is $J_v$.
The cell declares its own MDH robot first, so it can be copied and run on its own (lengths in m: l1 = 0.64, l2 = 0.2, l3 = 0.22, l4 = 0.095).


In [ ]:
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3

# 1) declare the robot with MDH (a = a_{i-1}, alpha = alpha_{i-1}, d = d_i, offset added to the joint variable)
robot = DHRobot([
    RevoluteMDH(a=0.0, alpha=0.0,     d=0.64,                    qlim=[-np.pi, np.pi]),   # joint 1 (R)
    RevoluteMDH(a=0.0, alpha=np.pi/2, d=-0.2,  offset=np.pi/2,   qlim=[-np.pi, np.pi]),   # joint 2 (R)
    PrismaticMDH(a=0.0, alpha=np.pi/2, offset=0.22,              qlim=[0.0, 0.3]),        # joint 3 (P), m
], name='my_mdh')
robot.tool = SE3.Tz(0.095)                  # l4: last joint -> tip
print(robot)

# 2) use it
Jv_rtb = robot.jacob0(q_val)[:3]
print('Jv =\n', np.round(Jv_rtb, 4))
print('det Jv =', round(np.linalg.det(Jv_rtb), 4))
print('M      =', round(robot.manipulability(q_val, axes='trans'), 4))

qd_rtb = np.linalg.solve(Jv_rtb, v_in)      # same as inv(Jv) @ v, but more accurate
print('q_dot =', np.round(qd_rtb, 4), '(rad/s, rad/s, m/s)')
print('check: |Jv q_dot - v| =', f'{np.linalg.norm(Jv_rtb @ qd_rtb - v_in):.1e}')